In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Folder path
folder = r"C:\Users\teren\OneDrive\Documents\Education\05._SpringBoard\09. Interview Prep\02. Relax Challenge"

# Load data
users = pd.read_csv(folder + r"\takehome_users.csv", encoding="latin-1")
engagement = pd.read_csv(folder + r"\takehome_user_engagement.csv")

# Clean dates
engagement["time_stamp"] = pd.to_datetime(engagement["time_stamp"])
engagement["date"] = engagement["time_stamp"].dt.date

# Keep unique login days per user
engagement = engagement[["user_id", "date"]].drop_duplicates().sort_values(["user_id", "date"])

# Function to mark adopted users
def adopted_user(dates):
    dates = list(pd.to_datetime(dates).sort_values())
    for i in range(len(dates) - 2):
        if (dates[i+2] - dates[i]).days <= 7:
            return 1
    return 0

# Build adopted flag
adopted = engagement.groupby("user_id")["date"].apply(adopted_user).reset_index()
adopted.columns = ["object_id", "adopted_user"]

# Merge with users table
df = users.merge(adopted, on="object_id", how="left")
df["adopted_user"] = df["adopted_user"].fillna(0).astype(int)

# Basic features
df["was_invited"] = df["invited_by_user_id"].notnull().astype(int)
org_size = df.groupby("org_id")["object_id"].count().rename("org_size")
df = df.merge(org_size, on="org_id", how="left")

# Select predictors
X = df[[
    "opted_in_to_mailing_list",
    "enabled_for_marketing_drip",
    "was_invited",
    "org_size",
    "creation_source"
]]

# One-hot encode creation source
X = pd.get_dummies(X, columns=["creation_source"], drop_first=True)
X["org_size"] = X["org_size"].fillna(X["org_size"].median())

y = df["adopted_user"]

# Scale and fit logistic regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(max_iter=1000)
model.fit(X_scaled, y)

# Show coefficients
results = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", ascending=False)

print("Adopted users:", df["adopted_user"].sum())
print("Adoption rate:", round(df["adopted_user"].mean() * 100, 2), "%")
print(results)

Adopted users: 1656
Adoption rate: 13.8 %
                              Feature  Coefficient
2                         was_invited     0.115642
7  creation_source_SIGNUP_GOOGLE_AUTH     0.081897
0            opted_in_to_mailing_list     0.019357
6              creation_source_SIGNUP     0.009908
1          enabled_for_marketing_drip     0.003513
4          creation_source_ORG_INVITE    -0.131050
5   creation_source_PERSONAL_PROJECTS    -0.230085
3                            org_size    -0.337631
